# 03 — Clean the data and split it by time

Full dataset (not the 10% sample): `data/interim/` → `data/processed/{train,val,test,items}.parquet`.

**Leakage rules**
- Every feature uses only data from before the cutoff.
- Fit scalers, encoders, TF-IDF and K-Means on train only; `.transform()` val and test.
- Never use metadata `average_rating` / `rating_number` (collected in 2023, they include future reviews). Use the `train_*` columns recomputed below.
- Never shuffle rows across time.

In [1]:
import sys

import pandas as pd

sys.path.insert(0, "../src")
from shoprank.config import load_config
from shoprank.data.clean import add_train_item_stats, clean_items, clean_reviews, k_core
from shoprank.data.split import eval_users, time_split

cfg = load_config("../config.yaml")
START, T1, T2, K = pd.Timestamp(cfg["min_date"]), cfg.t1, cfg.t2, cfg["k_core"]
OUT = cfg.path("processed")
OUT.mkdir(parents=True, exist_ok=True)

reviews = pd.read_parquet(cfg.path("interim") / "reviews.parquet")
items = pd.read_parquet(cfg.path("interim") / "items.parquet")
print(reviews.shape, items.shape)

(4624615, 8) (137269, 9)


## 1, 3, 4 — Clean reviews, define an interaction, keep a recent window

`clean_reviews` drops rows missing ids/rating and exact duplicates, keeps the latest review per (user, product), adds `liked` (rating ≥ 4) and keeps reviews from 2018 on.

In [2]:
reviews, counts = clean_reviews(reviews, START, cfg["liked_min_rating"])
pd.Series(counts)

raw                                      4624615
after dropna                             4624615
after exact dedup                        4570966
after keep-latest per (user, product)    4555500
from 2018-01-01                          2374766
dtype: int64

`liked` is the positive label; the 2018+ window keeps the catalogue current.

In [3]:
print(f"liked share: {reviews['liked'].mean():.1%}  rows from {START.date()}: {len(reviews):,}")

liked share: 73.0%  rows from 2018-01-01: 2,374,766


## 5 — 5-core filter

In [4]:
reviews = k_core(reviews, K)
counts[f"{K}-core"] = len(reviews)

assert reviews.groupby("user_id").size().min() >= K
assert reviews.groupby("parent_asin").size().min() >= K
print(pd.Series(counts).to_string())
print(f"users {reviews['user_id'].nunique():,}  products {reviews['parent_asin'].nunique():,}  "
      f"liked {reviews['liked'].mean():.1%}")

raw                                      4624615
after dropna                             4624615
after exact dedup                        4570966
after keep-latest per (user, product)    4555500
from 2018-01-01                          2374766
5-core                                    184567
users 24,243  products 9,999  liked 83.3%


## 6 — Split by time

The guide suggests T1 = 2023-06-01, T2 = 2023-08-01, but after the 5-core filter June–September 2023 holds only a few hundred reviews (collection ended Sept 2023, so the last months are nearly empty). T1 = 2022-07-01 and T2 = 2023-01-01 (chosen in Phase 2) give usable validation and test sets, each with one holiday peak.

In [5]:
print(reviews.set_index("timestamp").resample("MS").size().loc["2022-01":].to_string())

parts = time_split(reviews, T1, T2)
train, val, test = parts["train"], parts["val"], parts["test"]
for name, df in parts.items():
    print(f"{name:5} {len(df):>8,} rows ({len(df) / len(reviews):.1%})  users {df['user_id'].nunique():>6,}  "
          f"products {df['parent_asin'].nunique():>6,}  {df['timestamp'].min().date()} -> {df['timestamp'].max().date()}")

timestamp
2022-01-01    2637
2022-02-01    2120
2022-03-01    2429
2022-04-01    2083
2022-05-01    2095
2022-06-01    1844
2022-07-01    2354
2022-08-01    2824
2022-09-01    2845
2022-10-01    3793
2022-11-01    4995
2022-12-01    4516
2023-01-01    4531
2023-02-01    2930
2023-03-01    3447
2023-04-01    1951
2023-05-01     476
2023-06-01     172
2023-07-01     127
2023-08-01      81
2023-09-01       1
Freq: MS
train  149,524 rows (81.0%)  users 22,324  products  8,519  2018-01-01 -> 2022-06-30
val     21,327 rows (11.6%)  users  6,939  products  4,696  2022-07-01 -> 2022-12-31
test    13,716 rows (7.4%)  users  4,686  products  3,917  2023-01-01 -> 2023-09-02


## 7 — Which users to score

**warm** = seen in train and has ≥1 liked product in the window (main metric). **cold** = has ≥1 liked product in the window but no train history (tracked separately).

In [6]:
train_users = set(train["user_id"])
users_to_score = pd.concat([
    eval_users(val, train_users, "val"),
    eval_users(test, train_users, "test"),
], ignore_index=True)
print(users_to_score.groupby(["split", "group"]).size().unstack())

group  cold  warm
split            
test   1428  2686
val    1495  4549


## 2 — Clean metadata (+ product rating features from train only)

In [7]:
it, median_price = clean_items(items, reviews["parent_asin"])
it = add_train_item_stats(it, train)

assert it["parent_asin"].is_unique and set(reviews["parent_asin"]) <= set(it["parent_asin"])
print(f"median price used for gaps: ${median_price:.2f}  price_missing {it['price_missing'].mean():.1%}  "
      f"store unknown {(it['store'] == 'unknown').mean():.1%}  no train reviews {(it['train_rating_count'] == 0).mean():.1%}")
print(it.dtypes)
it[["parent_asin", "item_text"]].head(3)

median price used for gaps: $27.97  price_missing 24.3%  store unknown 0.2%  no train reviews 14.8%
parent_asin               str
title                     str
price                 float64
store                     str
categories             object
price_missing            int8
item_text                 str
platform                  str
train_rating_count      int32
train_avg_rating      float64
train_liked_rate      float64
dtype: object


,parent_asin,item_text
2,B00Z9TLVK0,nba 2k17 - early tip off edition - playstation...
7,B07H93H878,extremerate soft touch top shell front housing...
10,B07DL353F8,tales of vesperia - definitive edition - plays...


## 9 — Save

In [8]:
for name, df in parts.items():
    df.to_parquet(OUT / f"{name}.parquet", index=False)
it.to_parquet(OUT / "items.parquet", index=False)
users_to_score.to_parquet(OUT / "eval_users.parquet", index=False)

for name in ["train", "val", "test", "items", "eval_users"]:
    f = OUT / f"{name}.parquet"
    print(f"{f.name:20} {len(pd.read_parquet(f)):>8,} rows  {f.stat().st_size / 1e6:6.1f} MB")

train.parquet         149,524 rows    35.1 MB
val.parquet            21,327 rows     7.1 MB
test.parquet           13,716 rows     4.3 MB
items.parquet           9,999 rows     8.9 MB
eval_users.parquet     10,158 rows     0.2 MB
